# 02회 AICE Associate 모의고사 (회귀)
**시나리오: 배달 소요시간 예측**

배달 플랫폼 데이터 분석가로서, 주문 정보로 **배달 소요시간(분)을 예측하는 회귀 모델**을 만듭니다.
이번 회차는 **json 파일 읽기, 값 변환(map), 회귀 지표**가 포인트예요.

| 컬럼 | 설명 |
|---|---|
| order_id | 주문 번호 |
| distance_km | 배달 거리 |
| weather | 날씨 (맑음/흐림/비/눈) |
| time_slot | 시간대 (점심/저녁/심야/기타) |
| traffic | 교통 혼잡도 (낮음/보통/높음) |
| rider_age | 라이더 나이 |
| rider_rating | 라이더 평점 (결측 있음) |
| items | 주문 메뉴 개수 |
| store_type | 가게 종류 (한식/중식/치킨/카페) |
| **delivery_min** | **배달 소요시간 (타깃, 분)** |

## 0. 준비 (문제 아님, 실행만 하세요)

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(11)
n = 2500
dist = np.round(rng.gamma(2.5, 1.2, n), 1)
weather = rng.choice(['맑음','흐림','비','눈'], n, p=[0.5,0.25,0.18,0.07])
slot = rng.choice(['점심','저녁','심야','기타'], n, p=[0.3,0.35,0.1,0.25])
traffic = rng.choice(['낮음','보통','높음'], n, p=[0.35,0.4,0.25])
items = rng.integers(1, 7, n)
store = rng.choice(['한식','중식','치킨','카페'], n)
t = (12 + 5.5*dist + np.select([weather=='비', weather=='눈'], [6, 12], 0)
     + np.select([traffic=='보통', traffic=='높음'], [4, 10], 0)
     + np.select([slot=='저녁', slot=='점심'], [5, 3], 0) + 1.2*items
     + np.where(store=='치킨', 4, 0) + rng.normal(0, 4, n))
df = pd.DataFrame({'order_id': [f'D{i:05d}' for i in range(n)], 'distance_km': dist,
    'weather': weather, 'time_slot': slot, 'traffic': traffic,
    'rider_age': rng.integers(20, 60, n), 'rider_rating': np.round(rng.uniform(3, 5, n), 1),
    'items': items, 'store_type': store, 'delivery_min': np.round(t, 1)})
df.loc[rng.choice(n, 120, replace=False), 'rider_rating'] = np.nan
df.loc[rng.choice(n, 15, replace=False), 'delivery_min'] = 999
df.to_json('delivery_data.json', orient='records', force_ascii=False)
print('delivery_data.json 생성 완료')

## 1. 라이브러리 임포트

pandas(`pd`), numpy(`np`), scikit-learn(`sk`), seaborn(`sns`), matplotlib.pyplot(`plt`)을 별칭(alias)으로 임포트하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
import pandas as pd
import numpy as np
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
```

매 회차 1번은 똑같아요. 손이 기억할 때까지 직접 치세요.
</details>

## 2. JSON 데이터 불러오기

`delivery_data.json` 파일을 읽어 **`df`** 에 저장하고, 행과 열 개수(shape)를 출력하세요.

> 💡 힌트: pd.read_json

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df = pd.read_json('delivery_data.json')   # json 파일 읽기
print(df.shape)                           # (2500, 10)
df.head()
```

답: **(2500, 10)**

csv면 `read_csv`, json이면 `read_json`, 엑셀이면 `read_excel`. 시험에 json이 실제로 나와요.
</details>

## 3. 시각화 (1) 타깃 분포

seaborn **histplot** 으로 `delivery_min`의 분포를 그리세요.

그래프 오른쪽 끝에 **비정상적으로 큰 값**이 보이나요? 그 값은 대략 얼마인가요?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.histplot(data=df, x='delivery_min')
plt.show()

df['delivery_min'].max()   # 가장 큰 값 확인
```

답: **999분**. 배달에 16시간은 말이 안 되죠. 7번에서 지울 이상치예요. 숫자 분포는 histplot, 범주 개수는 countplot.
</details>

## 4. 시각화 (2) 범주별 비교

seaborn **boxplot** 으로 `weather`(x축)별 `delivery_min`(y축)을 비교하세요. 단, 999 때문에 그래프가 찌그러지니 `delivery_min < 200` 인 데이터만 사용하세요.

중앙값이 **가장 큰 날씨**는?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.boxplot(data=df[df['delivery_min'] < 200], x='weather', y='delivery_min')
plt.show()

df[df['delivery_min'] < 200].groupby('weather')['delivery_min'].median()   # 숫자로 확인
```

답: **눈**.

`groupby('기준열')['볼열'].median()`는 '날씨별로 묶어서 시간 중앙값'이라는 뜻이에요. 그래프 해석이 애매하면 groupby로 숫자를 직접 확인하면 확실해요.
</details>

## 5. 불필요한 컬럼 삭제

`order_id`를 삭제해 **`df1`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df1 = df.drop(columns=['order_id'])
```

주문 번호는 예측에 도움 안 되는 고유값이라 삭제해요. ID, 이름, 번호 같은 열은 거의 항상 삭제 대상.
</details>

## 6. 결측치 처리

`df1`에서 결측치가 있는 컬럼을 찾고, 그 결측치를 **평균값(mean)** 으로 채워 **`df2`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
print(df1.isnull().sum())    # rider_rating에 120개

df2 = df1.copy()
df2['rider_rating'] = df2['rider_rating'].fillna(df2['rider_rating'].mean())
print(df2.isnull().sum().sum())   # 전체 결측치 0개 확인
```

답: 결측 컬럼은 **rider_rating**.

`.sum().sum()`은 열별 합계를 한 번 더 더해서 전체 결측치 개수를 보는 방법이에요.
</details>

## 7. 이상치 제거

`df2`에서 `delivery_min`이 **200 이상인 행**을 제거해 **`df3`** 에 저장하고 행 개수를 출력하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df3 = df2[df2['delivery_min'] < 200]   # 200 미만만 남기기
print(len(df3))
```

답: **2485행** (15행 제거).

'~이상인 행을 제거' = '~미만인 행만 남기기'. 부등호 방향을 거꾸로 생각하면 쉬워요.
</details>

## 8. 값 변환 (순서가 있는 범주)

`traffic`은 순서가 있는 값이에요. **낮음 → 0, 보통 → 1, 높음 → 2** 로 바꿔 `df3`에 그대로 반영하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df3 = df3.copy()
df3['traffic'] = df3['traffic'].map({'낮음': 0, '보통': 1, '높음': 2})
df3['traffic'].value_counts()
```

`map`에 딕셔너리 `{원래값: 바꿀값}`를 넣어요. `replace`로 해도 돼요.

순서가 있는 범주(낮음<보통<높음)는 이렇게 숫자로, 순서가 없는 범주(한식, 중식)는 원핫 인코딩으로 처리하는 게 원칙이에요.
</details>

## 9. 원핫 인코딩

남은 문자 컬럼 `weather`, `time_slot`, `store_type`을 원핫 인코딩(`drop_first=True`)해 **`df_preset`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
cols = ['weather', 'time_slot', 'store_type']
df_preset = pd.get_dummies(df3, columns=cols, drop_first=True)
print(df_preset.shape)
```

답: **(2485, 15)**

9개 열 중 3개가 펼쳐지면서 늘어나요. 8번에서 traffic을 이미 숫자로 바꿨으니 여기엔 넣지 않아요.
</details>

## 10. 학습/검증 데이터 분리

`delivery_min`을 y, 나머지를 X로 하고 `test_size=0.2, random_state=42`로 분리하세요. (변수명 `X_train, X_test, y_train, y_test`)

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.model_selection import train_test_split

X = df_preset.drop('delivery_min', axis=1)
y = df_preset['delivery_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

**회귀에서는 stratify를 쓰지 않아요.** stratify는 '범주 비율 유지'라서 숫자가 연속인 타깃엔 쓸 수 없어요. 분류일 때만!
</details>

## 11. 스케일링 (MinMaxScaler)

**MinMaxScaler** 로 X_train과 X_test를 스케일링하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # train: fit + transform
X_test = scaler.transform(X_test)         # test: transform만
```

StandardScaler와 사용법이 100% 똑같아요. 이름만 바꾸면 됩니다.
</details>

## 12. 머신러닝 회귀 모델

다음 두 모델을 학습하세요.
- **LinearRegression** → `lr`
- **RandomForestRegressor** (`n_estimators=100, random_state=42`) → `rf`

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lr = LinearRegression()
lr.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
```

회귀 모델은 이름이 **Regressor** 로 끝나요 (LinearRegression만 예외). 분류는 **Classifier**.
</details>

## 13. 회귀 성능 평가

두 모델의 **MAE, MSE, R2** 를 출력하세요. MAE 기준으로 **더 좋은 모델**은?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

for name, m in [('LR', lr), ('RF', rf)]:
    pred = m.predict(X_test)
    print(name,
          'MAE:', round(mean_absolute_error(y_test, pred), 2),
          'MSE:', round(mean_squared_error(y_test, pred), 2),
          'R2:', round(r2_score(y_test, pred), 3))
```

답: **LinearRegression** (MAE 약 3.3 vs RF 약 4.2, R2도 0.91 vs 0.86). 이 데이터는 시간이 거리에 거의 직선으로 비례하게 만들어져서 단순한 선형 모델이 더 잘 맞아요. 복잡한 모델이 항상 이기는 건 아니에요!

**MAE, MSE는 낮을수록, R2는 높을수록 좋아요.** for문이 어려우면 모델마다 따로 세 줄씩 써도 정답이에요.
</details>

## 14. 딥러닝 회귀 모델 + 학습 곡선

Keras로 회귀 DNN을 만들고 학습한 뒤 loss 그래프를 그리세요.
- Dense(64, relu) → Dense(32, relu) → **회귀용 출력층**
- optimizer=`adam`, loss=`mse`, metrics=`['mae']`
- EarlyStopping(`monitor='val_loss', patience=10`), epochs=100, batch_size=32, `validation_split=0.2`
- 학습 결과는 `history`, 그 후 `loss`, `val_loss` 그래프

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)                          # 회귀: 숫자 하나, 활성화 함수 없음
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

es = EarlyStopping(monitor='val_loss', patience=10)
history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                    validation_split=0.2, callbacks=[es])

plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epochs'); plt.ylabel('MSE')
plt.legend(); plt.show()
```

회귀 출력층은 **`Dense(1)`** 만 쓰고 activation을 안 붙여요. 이게 가장 자주 나오는 함정이에요.

`validation_split=0.2`는 train의 20%를 떼서 검증에 쓰라는 뜻. `validation_data=(X_test, y_test)`로 직접 줘도 돼요. 문제에서 시키는 걸 쓰세요.
</details>

---
# 🏁 마무리 체크리스트
- 막히면 `함수이름?` 실행 또는 괄호 안에서 **Shift + Tab**
- `train_test_split` 반환 순서: **X_train, X_test, y_train, y_test**
- test 데이터는 **transform만** (fit 금지)
- 출력층과 loss 짝꿍 확인 (이진 sigmoid+binary_crossentropy / 다중 softmax+sparse_categorical_crossentropy / 회귀 Dense(1)+mse)
- 문제에서 정한 **변수명** 그대로 쓰기
- 다 풀었으면 정답 안 보고 **60분 타이머** 맞춰 한 번 더!